### Ce notebook permet de caractériser les segments issu de notre clustering

In [ ]:

 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
 
# 0. Chargement des données déjà clusterisées

CHEMIN_FICHIER = "../outputs/resume_segments_valid.csv"
 
rfm = pd.read_csv(CHEMIN_FICHIER)
 
print(f"Données chargées : {rfm.shape[0]} clients, {rfm['segment'].nunique()} segments")
 


In [ ]:
rfm.columns

### Nommage des segments et recommandation d'action

In [ ]:

# Nommage et recommandation d'action
#    Basé sur la position relative de chaque segment (au-dessus ou
#    en-dessous des médianes globales de récence/fréquence/montant)

med_r = rfm["recence_moyenne"].median()
med_f = rfm["frequence_moyenne"].median()
med_m = rfm["montant_moyen"].median()
 
def nommer_segment(row):
    recent = row["recence_moyenne"] <= med_r
    frequent = row["frequence_moyenne"] >= med_f
    gros_montant = row["montant_moyen"] >= med_m
 
    if recent and frequent and gros_montant:
        return "Client_fidele", "Fidéliser en priorité : programme VIP, avant-premières, remerciement personnalisé"
    elif not recent and not frequent:
        return "À risque / inactifs", "Campagne de réactivation urgente (offre de retour, email de relance)"
    elif recent and not frequent:
        return "Nouveaux prometteurs", "Encourager le 2e achat : onboarding, offre de bienvenue ciblée"
    elif not recent and gros_montant:
        return "Gros clients qui s'éloignent", "Contact prioritaire : ces clients avaient de la valeur, à ne pas perdre"
    else:
        return "Clients réguliers", "Maintenir l'engagement : newsletters, programme de fidélité standard"
 
rfm[["classe", "action_recommandee"]] = rfm.apply(
    lambda row: pd.Series(nommer_segment(row)), axis=1
)


### Tableau de synthèse final

In [ ]:
 
# 4. Tableau de synthèse final

tableau_final = rfm[[
    "segment", "effectif", "chiffre_affaires_segment", "part_chiffre_affaires_pct",
    "montant_moyen", "recence_moyenne", "action_recommandee"
]].round(1).sort_values("part_chiffre_affaires_pct", ascending=False)

tableau_final.columns = [
    "Segment", "Nb clients", "% du CA", "Indice de valeur",
    "Panier moyen (£)", "Récence moy. (j)", "Action recommandée"
]
 
print("\n=== Synthèse des segments pour décideurs ===\n")
print(tableau_final.to_string(index=False))
 
tableau_final.to_csv("../outputs/synthese_segments_decideurs.csv", index=False)
 

### Quelques graphiques d'illustration

In [ ]:


# 5. Graphique à bulles : lisible en 1 coup d'œil
#    x = récence (plus à gauche = plus récent = mieux)
#    y = fréquence (plus haut = plus fidèle)
#    taille des bulles = poids en % du CA

fig, ax = plt.subplots(figsize=(8, 6))
 
couleurs = plt.cm.viridis(np.linspace(0, 1, len(rfm)))
 
for i, row in rfm.iterrows():
    ax.scatter(
        row["recence_moyenne"], row["frequence_moyenne"],
        s=row["part_chiffre_affaires_pct"] * 40,  # taille proportionnelle au poids en CA
        alpha=0.7, color=couleurs[i], edgecolors="black", linewidth=1
    )
    ax.annotate(
        f"{row['segment']}\n({row['part_chiffre_affaires_pct']:.0f}% du CA)",
        (row["recence_moyenne"], row["frequence_moyenne"]),
        textcoords="offset points", xytext=(0, 12),
        ha="center", fontsize=9, fontweight="bold"
    )
 
ax.invert_xaxis()  # récence faible (= client actif) affichée à droite
ax.set_xlabel("Récence moyenne (jours depuis dernier achat) — inversé : actif → à droite")
ax.set_ylabel("Fréquence moyenne (nb commandes)")
ax.set_title("Cartographie des segments clients\n(taille des bulles = poids dans le CA total)")
plt.tight_layout()
plt.savefig("../figures/cartographie_segments.png", dpi=150)
plt.close()
 
print("\nFichiers générés :")
print("- synthese_segments_decideurs.csv (tableau à intégrer dans un rapport/slide)")
print("- cartographie_segments.png (visuel pour présentation)")
 

### Fin du pipeline